# Lesson 3: Chatbot Example

In this lesson, you will familiarize yourself with the chatbot example you will work on during this course. The example includes the tool definitions and execution, as well as the chatbot code. Make sure to interact with the chatbot at the end of this notebook.

## Import Libraries

```python

In [ ]:
import arxiv

In [ ]:
import json

In [ ]:
import os

from typing import List

from dotenv import load_dotenv

In [ ]:
import anthropic

```

## Tool Functions

```python

In [ ]:
PAPER_DIR = "papers"

```

```python

In [ ]:
def search_papers(topic: str, max_results: int = 5) -> List[str]:

"""

Search for papers on arXiv based on a topic and store their information.

Args:

topic: The topic to search for

max_results: Maximum number of results to retrieve (default: 5)

Returns:

List of paper IDs found in the search

"""

# Use arxiv to find the papers

client = arxiv.Client()

# Search for the most relevant articles matching the queried topic

search = arxiv.Search(

query = topic,

max_results = max_results,

sort_by = arxiv.SortCriterion.Relevance

)

papers = client.results(search)

# Create directory for this topic

In [ ]:
path = os.path.join(PAPER_DIR, topic.lower().replace(" ", "_"))

os.makedirs(path, exist_ok=True)

In [ ]:
file_path = os.path.join(path, "papers_info.json")

# Try to load existing papers info

try:

with open(file_path, "r") as json_file:

papers_info = json.load(json_file)

except (FileNotFoundError, json.JSONDecodeError):

papers_info = {}

# Process each paper and add to papers_info

In [ ]:
paper_ids = []

for paper in papers:

paper_ids.append(paper.get_short_id())

paper_info = {

'title': paper.title,

'authors': [author.name for author in paper.authors],

'summary': paper.summary,

'pdf_url': paper.pdf_url,

'published': str(paper.published.date())

}

In [ ]:
papers_info[paper.get_short_id()] = paper_info

# Save updated papers_info to json file

with open(file_path, "w") as json_file:

json.dump(papers_info, json_file, indent=2)

print(f"Results are saved in: {file_path}")

return paper_ids

```

```python

search_papers("computers")

```

```python

In [ ]:
def extract_info(paper_id: str) -> str:

"""

Search for information about a specific paper across all topic directories.

Args:

paper_id: The ID of the paper to look for

Returns:

JSON string with paper information if found, error message if not found

"""

for item in os.listdir(PAPER_DIR):

item_path = os.path.join(PAPER_DIR, item)

if os.path.isdir(item_path):

In [ ]:
file_path = os.path.join(item_path, "papers_info.json")

if os.path.isfile(file_path):

try:

with open(file_path, "r") as json_file:

papers_info = json.load(json_file)

if paper_id in papers_info:

return json.dumps(papers_info[paper_id], indent=2)

except (FileNotFoundError, json.JSONDecodeError) as e:

print(f"Error reading {file_path}: {str(e)}")

continue

return f"There's no saved information related to paper {paper_id}."

```

```python

extract_info('1312.3300v1')

```

## Tool Schema

```python

In [ ]:
tools = [

{

"name": "search_papers",

"description": "Search for papers on arXiv based on a topic and store their information.",

"input_schema": {

"type": "object",

"properties": {

"topic": {

"type": "string",

"description": "The topic to search for"

},

"max_results": {

"type": "integer",

"description": "Maximum number of results to retrieve",

"default": 5

}

},

"required": ["topic"]

}

},

{

"name": "extract_info",

"description": "Search for information about a specific paper across all topic directories.",

"input_schema": {

"type": "object",

"properties": {

"paper_id": {

"type": "string",

"description": "The ID of the paper to look for"

}

},

"required": ["paper_id"]

}

}

]

```

## Tool Mapping

```python

mapping_tool_function = {

"search_papers": search_papers,

"extract_info": extract_info

}

In [ ]:
def execute_tool(tool_name, tool_args):

In [ ]:
result = mapping_tool_function[tool_name](**tool_args)

if result is None:

In [ ]:
result = "The operation completed but didn't return any results."

elif isinstance(result, list):

result = ', '.join(result)

elif isinstance(result, dict):

# Convert dictionaries to formatted JSON strings

result = json.dumps(result, indent=2)

else:

# For any other type, convert using str()

result = str(result)

return result

```

!pip install langchain_anthropic

```python

from langchain_anthropic import ChatAnthropic

from langchain_core.tools import tool

```